<a href="https://colab.research.google.com/github/praveenkumar731/Expenses-Dashboard-FY2025/blob/main/Ecommerce_Sales_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TALLY DAY BOOK ANALYZER
# Google Colab + Python + Excel
#
# NO NARRATION REQUIREMENT
# ============================================================

import pandas as pd
import numpy as np
import re
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


# ============================================================
# 1. UPLOAD TALLY DAY BOOK
# ============================================================

print("=" * 70)
print("TALLY DAY BOOK ANALYZER")
print("=" * 70)

print("\nUpload your Tally Day Book Excel or CSV file")

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("\nSelected file:", file_name)


# ============================================================
# 2. READ EXCEL / CSV
# ============================================================

if file_name.lower().endswith(".csv"):

    df = pd.read_csv(
        file_name,
        low_memory=False
    )

else:

    df = pd.read_excel(
        file_name
    )


print("\nRecords loaded:", len(df))


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

print("\nColumns found:")
print(list(df.columns))


# ============================================================
# 4. COLUMN DETECTION FUNCTION
# ============================================================

def find_column(possible_names):

    normalized_columns = {}

    for col in df.columns:

        key = re.sub(
            r"[^a-z0-9]",
            "",
            str(col).lower()
        )

        normalized_columns[key] = col


    # Exact matching
    for name in possible_names:

        key = re.sub(
            r"[^a-z0-9]",
            "",
            name.lower()
        )

        if key in normalized_columns:

            return normalized_columns[key]


    # Partial matching
    for col in df.columns:

        clean_col = re.sub(
            r"[^a-z0-9]",
            "",
            str(col).lower()
        )

        for name in possible_names:

            clean_name = re.sub(
                r"[^a-z0-9]",
                "",
                name.lower()
            )

            if (
                clean_name in clean_col
                or clean_col in clean_name
            ):

                return col


    return None


# ============================================================
# 5. DETECT TALLY COLUMNS
# ============================================================

date_col = find_column([
    "Date",
    "Voucher Date",
    "Transaction Date"
])

voucher_no_col = find_column([
    "Voucher No",
    "Voucher Number",
    "Vch No",
    "Vch Number",
    "Voucher"
])

voucher_type_col = find_column([
    "Voucher Type",
    "Vch Type",
    "Type"
])

particulars_col = find_column([
    "Particulars",
    "Particular",
    "Ledger",
    "Ledger Name",
    "Account"
])

debit_col = find_column([
    "Debit",
    "Debit Amount",
    "Dr",
    "Dr Amount"
])

credit_col = find_column([
    "Credit",
    "Credit Amount",
    "Cr",
    "Cr Amount"
])

amount_col = find_column([
    "Amount",
    "Transaction Amount",
    "Value"
])


# ============================================================
# 6. SHOW DETECTED COLUMNS
# ============================================================

print("\n")
print("=" * 70)
print("DETECTED COLUMNS")
print("=" * 70)

print("Date              :", date_col)
print("Voucher Number    :", voucher_no_col)
print("Voucher Type      :", voucher_type_col)
print("Particulars/Ledger:", particulars_col)
print("Debit             :", debit_col)
print("Credit            :", credit_col)
print("Amount            :", amount_col)


# ============================================================
# 7. DATE PROCESSING
# ============================================================

if date_col:

    df["Analysis Date"] = pd.to_datetime(
        df[date_col],
        errors="coerce",
        dayfirst=True
    )

else:

    df["Analysis Date"] = pd.NaT


df["Year"] = df["Analysis Date"].dt.year

df["Month"] = (
    df["Analysis Date"]
    .dt.to_period("M")
    .astype(str)
)

df["Month Name"] = (
    df["Analysis Date"]
    .dt.strftime("%b-%Y")
)

df["Day"] = (
    df["Analysis Date"]
    .dt.date
)

df["Day Name"] = (
    df["Analysis Date"]
    .dt.day_name()
)


# ============================================================
# 8. CLEAN NUMBER FUNCTION
# ============================================================

def clean_number(series):

    return (
        series
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("Rs", "", regex=False)
        .str.replace(" ", "", regex=False)
        .replace(
            ["", "-", "nan", "None"],
            np.nan
        )
        .astype(float)
    )


# ============================================================
# 9. DEBIT
# ============================================================

if debit_col:

    df["Debit Amount"] = (
        clean_number(
            df[debit_col]
        )
        .fillna(0)
    )

else:

    df["Debit Amount"] = 0


# ============================================================
# 10. CREDIT
# ============================================================

if credit_col:

    df["Credit Amount"] = (
        clean_number(
            df[credit_col]
        )
        .fillna(0)
    )

else:

    df["Credit Amount"] = 0


# ============================================================
# 11. TRANSACTION AMOUNT
# ============================================================

if amount_col:

    df["Transaction Amount"] = (
        clean_number(
            df[amount_col]
        )
        .fillna(0)
    )

else:

    df["Transaction Amount"] = (
        df["Debit Amount"]
        +
        df["Credit Amount"]
    )


# ============================================================
# 12. VOUCHER TYPE
# ============================================================

if voucher_type_col:

    df["Voucher Type Analysis"] = (
        df[voucher_type_col]
        .astype(str)
        .str.strip()
    )

else:

    df["Voucher Type Analysis"] = "Unknown"


# ============================================================
# 13. LEDGER / PARTICULARS
# ============================================================

if particulars_col:

    df["Ledger Analysis"] = (
        df[particulars_col]
        .astype(str)
        .str.strip()
    )

else:

    df["Ledger Analysis"] = "Unknown"


# ============================================================
# REPORT 1
# EXECUTIVE SUMMARY
# ============================================================

total_transactions = len(df)

total_debit = df[
    "Debit Amount"
].sum()

total_credit = df[
    "Credit Amount"
].sum()

total_transaction_value = df[
    "Transaction Amount"
].sum()

average_transaction = (
    df["Transaction Amount"].mean()
    if len(df) > 0
    else 0
)

highest_transaction = (
    df["Transaction Amount"].max()
    if len(df) > 0
    else 0
)

lowest_transaction = (
    df["Transaction Amount"].min()
    if len(df) > 0
    else 0
)

unique_ledgers = (
    df["Ledger Analysis"].nunique()
)

unique_voucher_types = (
    df["Voucher Type Analysis"].nunique()
)

if voucher_no_col:

    unique_vouchers = (
        df[voucher_no_col]
        .astype(str)
        .nunique()
    )

else:

    unique_vouchers = 0


summary = pd.DataFrame({

    "Metric": [

        "Total Transactions",
        "Unique Vouchers",
        "Unique Ledgers",
        "Voucher Types",
        "Total Debit",
        "Total Credit",
        "Total Transaction Value",
        "Average Transaction",
        "Highest Transaction",
        "Lowest Transaction"

    ],

    "Value": [

        total_transactions,
        unique_vouchers,
        unique_ledgers,
        unique_voucher_types,
        total_debit,
        total_credit,
        total_transaction_value,
        average_transaction,
        highest_transaction,
        lowest_transaction

    ]

})


# ============================================================
# REPORT 2
# VOUCHER SUMMARY
# ============================================================

voucher_summary = (

    df
    .groupby(
        "Voucher Type Analysis"
    )
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Total_Amount=(
            "Transaction Amount",
            "sum"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        )

    )
    .reset_index()
    .sort_values(
        "Total_Amount",
        ascending=False
    )

)


# ============================================================
# REPORT 3
# LEDGER SUMMARY
# ============================================================

ledger_summary = (

    df
    .groupby(
        "Ledger Analysis"
    )
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Total_Amount=(
            "Transaction Amount",
            "sum"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        )

    )
    .reset_index()
    .sort_values(
        "Total_Amount",
        ascending=False
    )

)


# ============================================================
# REPORT 4
# MONTHLY SUMMARY
# ============================================================

monthly_summary = (

    df
    .groupby("Month")
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        ),

        Transaction_Value=(
            "Transaction Amount",
            "sum"
        )

    )
    .reset_index()

)


# ============================================================
# REPORT 5
# DAILY SUMMARY
# ============================================================

daily_summary = (

    df
    .groupby("Day")
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        ),

        Transaction_Value=(
            "Transaction Amount",
            "sum"
        )

    )
    .reset_index()

)


# ============================================================
# REPORT 6
# TOP 50 TRANSACTIONS
# ============================================================

top_transactions = (

    df
    .sort_values(
        "Transaction Amount",
        ascending=False
    )
    .head(50)

)


# ============================================================
# REPORT 7
# LARGE TRANSACTIONS
# ============================================================

if len(df) > 0:

    percentile_95 = (
        df["Transaction Amount"]
        .quantile(0.95)
    )

    # Minimum threshold ₹1 lakh
    large_transaction_limit = max(
        percentile_95,
        100000
    )

else:

    large_transaction_limit = 100000


large_transactions = (

    df[
        df["Transaction Amount"]
        >= large_transaction_limit
    ]

    .sort_values(
        "Transaction Amount",
        ascending=False
    )

)


# ============================================================
# REPORT 8
# CASH & BANK ANALYSIS
# ============================================================

cash_bank_keywords = [

    "cash",
    "bank",
    "hdfc",
    "icici",
    "sbi",
    "axis",
    "kotak",
    "yes bank",
    "indusind",
    "union bank",
    "bob",
    "bank of baroda",
    "pnb",
    "canara"

]


cash_bank_pattern = "|".join(

    re.escape(keyword)

    for keyword in cash_bank_keywords

)


cash_bank_data = df[
    df["Ledger Analysis"]
    .str.lower()
    .str.contains(
        cash_bank_pattern,
        na=False
    )
]


cash_bank_summary = (

    cash_bank_data
    .groupby(
        "Ledger Analysis"
    )
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        ),

        Total=(
            "Transaction Amount",
            "sum"
        )

    )
    .reset_index()
    .sort_values(
        "Total",
        ascending=False
    )

)


# ============================================================
# REPORT 9
# EXPENSE ANALYSIS
# ============================================================

expense_keywords = [

    "expense",
    "salary",
    "rent",
    "electricity",
    "telephone",
    "internet",
    "travel",
    "conveyance",
    "printing",
    "stationery",
    "repairs",
    "maintenance",
    "office",
    "advertisement",
    "advertising",
    "professional fees",
    "bank charges",
    "courier",
    "insurance"

]


expense_pattern = "|".join(

    re.escape(keyword)

    for keyword in expense_keywords

)


expense_data = df[

    df["Ledger Analysis"]
    .str.lower()
    .str.contains(
        expense_pattern,
        na=False
    )

]


expense_summary = (

    expense_data
    .groupby(
        "Ledger Analysis"
    )
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        ),

        Total=(
            "Transaction Amount",
            "sum"
        )

    )
    .reset_index()
    .sort_values(
        "Total",
        ascending=False
    )

)


# ============================================================
# REPORT 10
# PARTY / LEDGER ANALYSIS
# ============================================================

party_analysis = (

    df
    .groupby(
        "Ledger Analysis"
    )
    .agg(

        Transactions=(
            "Transaction Amount",
            "count"
        ),

        Debit=(
            "Debit Amount",
            "sum"
        ),

        Credit=(
            "Credit Amount",
            "sum"
        ),

        Total=(
            "Transaction Amount",
            "sum"
        )

    )
    .reset_index()
    .sort_values(
        "Total",
        ascending=False
    )
    .head(100)

)


# ============================================================
# REPORT 11
# DUPLICATE VOUCHERS
# ============================================================

if voucher_no_col:

    voucher_series = (

        df[voucher_no_col]
        .astype(str)
        .str.strip()

    )

    duplicate_mask = (

        voucher_series
        .duplicated(
            keep=False
        )

    )

    duplicate_vouchers = (

        df[
            duplicate_mask
        ]
        .sort_values(
            by=voucher_no_col
        )

    )

else:

    duplicate_vouchers = (
        pd.DataFrame()
    )


# ============================================================
# REPORT 12
# VOUCHER NUMBER GAPS
# ============================================================

voucher_gaps = pd.DataFrame()

if voucher_no_col:

    numeric_vouchers = pd.to_numeric(

        df[voucher_no_col],

        errors="coerce"

    ).dropna()


    numeric_vouchers = sorted(

        numeric_vouchers
        .astype(int)
        .unique()

    )


    if len(numeric_vouchers) > 1:

        minimum = min(
            numeric_vouchers
        )

        maximum = max(
            numeric_vouchers
        )

        complete_range = set(
            range(
                minimum,
                maximum + 1
            )
        )

        existing_numbers = set(
            numeric_vouchers
        )

        missing_numbers = sorted(

            complete_range
            -
            existing_numbers

        )


        voucher_gaps = pd.DataFrame({

            "Missing Voucher Number":
                missing_numbers

        })


# ============================================================
# REPORT 13
# ROUND AMOUNT TRANSACTIONS
# ============================================================

round_amount_transactions = df[

    (
        df["Transaction Amount"]
        > 0
    )

    &

    (
        df["Transaction Amount"]
        % 1000
        == 0
    )

].sort_values(

    "Transaction Amount",
    ascending=False

)


# ============================================================
# REPORT 14
# WEEKEND TRANSACTIONS
# ============================================================

weekend_transactions = df[

    df["Day Name"]
    .isin([
        "Saturday",
        "Sunday"
    ])

].copy()


# ============================================================
# REPORT 15
# DEBIT / CREDIT ANALYSIS
# ============================================================

debit_credit_summary = pd.DataFrame({

    "Metric": [

        "Total Debit",
        "Total Credit",
        "Debit Transactions",
        "Credit Transactions",
        "Debit-Credit Difference"

    ],

    "Value": [

        total_debit,

        total_credit,

        (
            df["Debit Amount"]
            > 0
        ).sum(),

        (
            df["Credit Amount"]
            > 0
        ).sum(),

        total_debit
        -
        total_credit

    ]

})


# ============================================================
# REPORT 16
# AUDIT EXCEPTIONS
# ============================================================

audit_list = []


# ------------------------------------------------------------
# A. Large transactions
# ------------------------------------------------------------

for index, row in large_transactions.iterrows():

    audit_list.append({

        "Exception Type":
            "Large Transaction",

        "Date":
            row["Analysis Date"],

        "Voucher Type":
            row["Voucher Type Analysis"],

        "Ledger":
            row["Ledger Analysis"],

        "Amount":
            row["Transaction Amount"],

        "Reason":
            f"Amount exceeds ₹{large_transaction_limit:,.0f}"

    })


# ------------------------------------------------------------
# B. Duplicate vouchers
# ------------------------------------------------------------

if len(duplicate_vouchers) > 0:

    for index, row in duplicate_vouchers.iterrows():

        audit_list.append({

            "Exception Type":
                "Duplicate Voucher",

            "Date":
                row["Analysis Date"],

            "Voucher Type":
                row["Voucher Type Analysis"],

            "Ledger":
                row["Ledger Analysis"],

            "Amount":
                row["Transaction Amount"],

            "Reason":
                "Voucher number appears multiple times"

        })


# ------------------------------------------------------------
# C. Round amount transactions
# ------------------------------------------------------------

for index, row in round_amount_transactions.iterrows():

    audit_list.append({

        "Exception Type":
            "Round Amount",

        "Date":
            row["Analysis Date"],

        "Voucher Type":
            row["Voucher Type Analysis"],

        "Ledger":
            row["Ledger Analysis"],

        "Amount":
            row["Transaction Amount"],

        "Reason":
            "Transaction amount is exactly divisible by ₹1,000"

    })


# ------------------------------------------------------------
# D. Weekend transactions
# ------------------------------------------------------------

for index, row in weekend_transactions.iterrows():

    audit_list.append({

        "Exception Type":
            "Weekend Transaction",

        "Date":
            row["Analysis Date"],

        "Voucher Type":
            row["Voucher Type Analysis"],

        "Ledger":
            row["Ledger Analysis"],

        "Amount":
            row["Transaction Amount"],

        "Reason":
            "Transaction recorded on Saturday/Sunday"

    })


# ------------------------------------------------------------
# E. Missing date
# ------------------------------------------------------------

missing_date = df[
    df["Analysis Date"].isna()
]


for index, row in missing_date.iterrows():

    audit_list.append({

        "Exception Type":
            "Missing Date",

        "Date": "",

        "Voucher Type":
            row["Voucher Type Analysis"],

        "Ledger":
            row["Ledger Analysis"],

        "Amount":
            row["Transaction Amount"],

        "Reason":
            "Invalid or missing transaction date"

    })


# Create DataFrame

audit_exceptions = pd.DataFrame(
    audit_list
)


# ============================================================
# REPORT 17
# DATA QUALITY
# ============================================================

data_quality = pd.DataFrame({

    "Check": [

        "Total Records",

        "Missing Dates",

        "Duplicate Voucher Records",

        "Voucher Number Gaps",

        "Weekend Transactions",

        "Round Amount Transactions",

        "Large Transactions"

    ],

    "Count": [

        len(df),

        len(missing_date),

        len(duplicate_vouchers),

        len(voucher_gaps),

        len(weekend_transactions),

        len(round_amount_transactions),

        len(large_transactions)

    ]

})


# ============================================================
# REPORT 18
# TOP 20 LEDGERS
# ============================================================

top_20_ledgers = (

    ledger_summary
    .head(20)

)


# ============================================================
# REPORT 19
# TOP 20 EXPENSES
# ============================================================

top_20_expenses = (

    expense_summary
    .head(20)

)


# ============================================================
# REPORT 20
# CREATE EXCEL REPORT
# ============================================================

output_file = (
    "Tally_Day_Book_Analyzer.xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:


    summary.to_excel(
        writer,
        sheet_name="Executive Summary",
        index=False
    )


    voucher_summary.to_excel(
        writer,
        sheet_name="Voucher Summary",
        index=False
    )


    ledger_summary.to_excel(
        writer,
        sheet_name="Ledger Summary",
        index=False
    )


    monthly_summary.to_excel(
        writer,
        sheet_name="Monthly Summary",
        index=False
    )


    daily_summary.to_excel(
        writer,
        sheet_name="Daily Summary",
        index=False
    )


    top_transactions.to_excel(
        writer,
        sheet_name="Top Transactions",
        index=False
    )


    large_transactions.to_excel(
        writer,
        sheet_name="Large Transactions",
        index=False
    )


    cash_bank_summary.to_excel(
        writer,
        sheet_name="Cash Bank",
        index=False
    )


    expense_summary.to_excel(
        writer,
        sheet_name="Expense Analysis",
        index=False
    )


    party_analysis.to_excel(
        writer,
        sheet_name="Party Analysis",
        index=False
    )


    duplicate_vouchers.to_excel(
        writer,
        sheet_name="Duplicate Vouchers",
        index=False
    )


    voucher_gaps.to_excel(
        writer,
        sheet_name="Voucher Gaps",
        index=False
    )


    round_amount_transactions.to_excel(
        writer,
        sheet_name="Round Amount",
        index=False
    )


    weekend_transactions.to_excel(
        writer,
        sheet_name="Weekend Transactions",
        index=False
    )


    debit_credit_summary.to_excel(
        writer,
        sheet_name="Debit Credit",
        index=False
    )


    audit_exceptions.to_excel(
        writer,
        sheet_name="Audit Exceptions",
        index=False
    )


    data_quality.to_excel(
        writer,
        sheet_name="Data Quality",
        index=False
    )


    top_20_ledgers.to_excel(
        writer,
        sheet_name="Top 20 Ledgers",
        index=False
    )


    top_20_expenses.to_excel(
        writer,
        sheet_name="Top 20 Expenses",
        index=False
    )


    df.to_excel(
        writer,
        sheet_name="Original Data",
        index=False
    )


# ============================================================
# 21. FORMAT EXCEL
# ============================================================

wb = load_workbook(
    output_file
)


for ws in wb.worksheets:

    # Freeze first row
    ws.freeze_panes = "A2"


    # Header formatting
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center"
        )


    # Auto column width
    for column_cells in ws.columns:

        max_length = 0

        column_letter = (
            get_column_letter(
                column_cells[0].column
            )
        )


        for cell in column_cells:

            try:

                max_length = max(
                    max_length,
                    len(
                        str(
                            cell.value
                        )
                    )
                )

            except:

                pass


        ws.column_dimensions[
            column_letter
        ].width = min(
            max_length + 2,
            40
        )


# Save workbook

wb.save(
    output_file
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("ANALYSIS COMPLETED")
print("=" * 70)

print(
    f"\nTotal Transactions : {total_transactions:,}"
)

print(
    f"Total Debit        : ₹{total_debit:,.2f}"
)

print(
    f"Total Credit       : ₹{total_credit:,.2f}"
)

print(
    f"Highest Transaction: ₹{highest_transaction:,.2f}"
)

print(
    f"Large Transaction  : ₹{large_transaction_limit:,.2f}+"
)

print(
    f"Duplicate Vouchers : {len(duplicate_vouchers):,}"
)

print(
    f"Voucher Gaps       : {len(voucher_gaps):,}"
)

print(
    f"Weekend Entries    : {len(weekend_transactions):,}"
)

print(
    f"Round Amounts      : {len(round_amount_transactions):,}"
)

print(
    f"Audit Exceptions   : {len(audit_exceptions):,}"
)

print("\nReport created:")

print(
    output_file
)


# ============================================================
# 23. DOWNLOAD
# ============================================================

files.download(
    output_file
)


TALLY DAY BOOK ANALYZER

Upload your Tally Day Book Excel or CSV file


Saving Tally_Day_Book_Analyzer.xlsx to Tally_Day_Book_Analyzer.xlsx

Selected file: Tally_Day_Book_Analyzer.xlsx

Records loaded: 10

Columns found:
['Metric', 'Value']


DETECTED COLUMNS
Date              : None
Voucher Number    : None
Voucher Type      : None
Particulars/Ledger: None
Debit             : None
Credit            : None
Amount            : Value


ANALYSIS COMPLETED

Total Transactions : 10
Total Debit        : ₹0.00
Total Credit       : ₹0.00
Highest Transaction: ₹354,270,795.52
Large Transaction  : ₹354,270,795.52+
Duplicate Vouchers : 0
Voucher Gaps       : 0
Weekend Entries    : 0
Round Amounts      : 1
Audit Exceptions   : 13

Report created:
Tally_Day_Book_Analyzer.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>